# Alex Atkins huck reconciliation
The 2026 Connections chart defaults to regular season. Compare that total with all-season Shown Space counts. Uses the preserved September 29 snapshot, not a fresh fetch.

In [1]:
import json, math
from pathlib import Path
project = Path.cwd()
if not (project / 'src/data.json').exists():
    project = project.parent
snapshot = json.loads((project / 'src/data.json').read_text())
throws = [e for e in snapshot['queries']['events']['rows']
          if e['thrower'] == 'aatkins' and e['team'] == 'empire' and not e['stall']]
hucks = [e for e in throws if all(isinstance(e.get(k), (int, float)) for k in ('tx', 'ty', 'rx', 'ry'))
         and math.hypot(e['rx'] - e['tx'], e['ry'] - e['ty']) >= 40
         and e['ry'] - e['ty'] >= 30]
assert len(hucks) == len({e['id'] for e in hucks})
rows = []
for phase in ['Regular season', 'Playoffs', 'All 2026']:
    selected = [e for e in hucks if phase == 'All 2026' or e['phase'] == phase]
    rows.append({'phase': phase, 'attempts': len(selected),
                 'completions': sum(e['complete'] for e in selected),
                 'drops': sum(e['drop'] for e in selected)})
leaderboard = json.loads((project / 'data/source-api/leaderboard.json').read_text())
atkins = next(p for p in leaderboard['data'] if p['PlayerID'] == 'aatkins')
assert rows[-1]['attempts'] == atkins['hucks_thrown'] == 53
assert rows[-1]['completions'] == atkins['hucks_completed'] == 45
assert rows[-1]['drops'] == atkins['hucks_dropped'] == 2
assert abs(45 / (53 - 2) - atkins['huck_completion_percentage']) < 1e-12
print(json.dumps(rows, indent=2))
print('Shown Space HuCP: 45 / (53 - 2) = 88.2353%')


[
  {
    "phase": "Regular season",
    "attempts": 35,
    "completions": 28,
    "drops": 2
  },
  {
    "phase": "Playoffs",
    "attempts": 18,
    "completions": 17,
    "drops": 0
  },
  {
    "phase": "All 2026",
    "attempts": 53,
    "completions": 45,
    "drops": 2
  }
]
Shown Space HuCP: 45 / (53 - 2) = 88.2353%
